# 🏆 Notebook 6 — GOAT Analysis
**Formula 1 ML Analytics Project**

**Question answered**: *Who is the Greatest F1 Driver of All Time?*

## Methodology: 5 Lenses

| Weight | Lens | Description |
|--------|------|-------------|
| 20% | Raw Statistics | Era-normalized wins, poles, podiums, championships |
| 30% | Teammate Comparison | H2H comparison in same car (removes car advantage) |
| 25% | ML "Same Car" Test | Place all drivers in median car, simulate season |
| 15% | Adaptability Score | Teams won with, regulation changes, circuit variety |
| 10% | Dominance Index | Peak win rate, championship gaps, consecutive wins |

**GOAT Score** = Σ(weight × normalized_lens_score)

## Candidates
Lewis Hamilton, Michael Schumacher, Max Verstappen, Sebastian Vettel,
Alain Prost, Ayrton Senna, Juan Manuel Fangio, Jim Clark, Jackie Stewart,
Niki Lauda, Fernando Alonso


In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

sys.path.insert(0, "..")
DATA_PATH = "../data/processed/"
MODEL_PATH = "../models/"

# Load data
master_file = os.path.join(DATA_PATH, "master_df.csv")
feat_file   = os.path.join(DATA_PATH, "featured_df.csv")

if not os.path.exists(feat_file):
    print("⚠️  Run notebooks 01 and 02 first.")
else:
    master_df   = pd.read_csv(master_file, low_memory=False) if os.path.exists(master_file) else None
    featured_df = pd.read_csv(feat_file, low_memory=False)
    print(f"✅ master_df: {master_df.shape if master_df is not None else 'not loaded'}")
    print(f"✅ featured_df: {featured_df.shape}")

# Load race winner model
try:
    import joblib
    model_file = os.path.join(MODEL_PATH, "race_winner_model.pkl")
    race_model = joblib.load(model_file) if os.path.exists(model_file) else None
    print(f"Race model: {'✅ loaded' if race_model else '⚠️  not found'}")
except:
    race_model = None


In [ ]:
from src.goat_analysis import (
    compute_raw_stats,
    compute_teammate_comparison,
    compute_same_car_ml_score,
    compute_adaptability_score,
    compute_dominance_index,
    compute_goat_scores,
    run_goat_analysis
)

GOAT_CANDIDATES = [
    'Lewis Hamilton', 'Michael Schumacher', 'Max Verstappen', 'Sebastian Vettel',
    'Alain Prost', 'Ayrton Senna', 'Juan Manuel Fangio', 'Jim Clark',
    'Jackie Stewart', 'Niki Lauda', 'Fernando Alonso'
]
print(f"GOAT candidates ({len(GOAT_CANDIDATES)}):")
for c in GOAT_CANDIDATES:
    print(f"  - {c}")


In [ ]:
# === LENS 1: Raw Era-Normalized Statistics ===
print("\n📊 Computing Lens 1: Raw Statistics (era-normalized)...")
raw_stats = compute_raw_stats(master_df or featured_df, featured_df, candidates=GOAT_CANDIDATES)
print(raw_stats[['driver_name','total_wins','total_races','win_rate_pct',
                  'total_championships','normalized_wins']].to_string(index=False))


In [ ]:
# === LENS 2: Teammate Comparison ===
print("\n🤝 Computing Lens 2: Teammate Comparison...")
teammate_df = compute_teammate_comparison(master_df or featured_df, featured_df, candidates=GOAT_CANDIDATES)
cols = ['driver_name','career_qual_h2h_pct','career_race_h2h_pct',
        'career_avg_teammate_race_delta','teammate_score']
available = [c for c in cols if c in teammate_df.columns]
print(teammate_df[available].to_string(index=False))


In [ ]:
# === LENS 3: ML Same-Car Test ===
print("\n🔬 Computing Lens 3: ML Same-Car Test...")
print("(Simulating each driver in a median-performance car — may take a few minutes)")
same_car_df = compute_same_car_ml_score(featured_df, candidates=GOAT_CANDIDATES, model=race_model)
cols = ['driver_name','simulated_season_points','simulated_wins','same_car_score']
available = [c for c in cols if c in same_car_df.columns]
print(same_car_df[available].to_string(index=False))


In [ ]:
# === LENS 4: Adaptability Score ===
print("\n🌍 Computing Lens 4: Adaptability Score...")
adapt_df = compute_adaptability_score(master_df or featured_df, featured_df, candidates=GOAT_CANDIDATES)
cols = ['driver_name','num_teams_won_with','num_circuits_won_at','regulation_era_wins','adaptability_score']
available = [c for c in cols if c in adapt_df.columns]
print(adapt_df[available].to_string(index=False))


In [ ]:
# === LENS 5: Dominance Index ===
print("\n👑 Computing Lens 5: Dominance Index...")
dom_df = compute_dominance_index(master_df or featured_df, featured_df, candidates=GOAT_CANDIDATES)
cols = ['driver_name','peak_3yr_win_rate','consecutive_wins_record','dominant_seasons_count','dominance_score']
available = [c for c in cols if c in dom_df.columns]
print(dom_df[available].to_string(index=False))


In [ ]:
# === FINAL GOAT SCORE ===
print("\n🏆 Computing Final GOAT Scores...")
goat_df = compute_goat_scores(raw_stats, teammate_df, same_car_df, adapt_df, dom_df)

print("\n" + "="*75)
print("  🏆  F1 GREATEST OF ALL TIME — FINAL RANKINGS  🏆")
print("="*75)
display_cols = ['driver_name','normalized_stats_score','teammate_score','same_car_score',
                'adaptability_score','dominance_score','GOAT_Score']
available = [c for c in display_cols if c in goat_df.columns]
print(goat_df[available].to_string(index=False))

# Save GOAT results
goat_df.to_csv(os.path.join(DATA_PATH, "goat_results.csv"), index=False)
print(f"\n✅ GOAT results saved to {DATA_PATH}goat_results.csv")


In [ ]:
# Visualize GOAT radar chart
from src.visualizations import plot_goat_radar, plot_teammate_comparison
plot_goat_radar(goat_df)
plot_teammate_comparison(teammate_df)


In [ ]:
# Bar chart of final GOAT scores
if 'GOAT_Score' in goat_df.columns:
    df_plot = goat_df.sort_values('GOAT_Score', ascending=True)
    colors = ['#FFD700' if i == len(df_plot)-1 else '#C0C0C0' if i == len(df_plot)-2 
              else '#CD7F32' if i == len(df_plot)-3 else '#888888' 
              for i in range(len(df_plot))]
    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(df_plot['driver_name'], df_plot['GOAT_Score'], color=colors)
    ax.set_xlabel('GOAT Score (0–100)', fontsize=12)
    ax.set_title('F1 Greatest of All Time — Final Composite Score', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 110)
    for bar, score in zip(bars, df_plot['GOAT_Score']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{score:.1f}', va='center', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_PATH, "goat_rankings.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n🥇 GOAT WINNER: {goat_df.iloc[0]['driver_name']} ({goat_df.iloc[0]['GOAT_Score']:.1f}/100)")
